# Brazil adaptation mock API

Builds example responses from the input tables in `../sample/` (gitignored) and the city order in `../city-action-order.csv`. Four endpoints:

| Response | Files | What it holds |
|---|---|---|
| `GET /api/v1/cities/{locode}/adaptation-actions` | `city-adaptation-actions-BR-CXL.json` (Caxias do Sul) | The actions ordered for one city, with their scores and attributes, and the city's finance context |
| `GET /api/v1/adaptation-actions/{action_id}` | `adaptation-action-{action_id}.json` | One action with everything behind it: impact links, legal norms, cost parts, co-benefits, its supporting actions (pre- or co-action), finance programmes, and the top 5 policy documents and past projects with their totals |
| `GET /api/v1/adaptation-actions/{action_id}/policy-documents?page=&page_size=` | `adaptation-action-{action_id}-policy-documents.json` | Every policy document with graded evidence for the action, best passage per document, strongest first, paged |
| `GET /api/v1/adaptation-actions/{action_id}/past-projects?page=&page_size=` | `adaptation-action-{action_id}-past-projects.json` | Every past project matched to the action, high matches first then newest, paged |

Supporting actions (no Direct link to any risk) are not ordered, so they are not in the city list. They are shown on the actions they are a pre- or co-action of.

The city list is read from `../city-action-order.csv` (built by `../build_sample_data.ipynb`). The action detail is the same for every city: everything city-specific is in the city list. Every field is defined in `api-fields.csv`.

City risk is already served by `GET /api/v1/cities/{locode}/climate-risk/adapta`, so it has no mock here. Responses use the `meta` and `data` envelope of the existing global API. `lang` picks the name and description language (`en` or `pt`).

In [1]:
import csv, json
from pathlib import Path

DATA = Path('../sample')
ORDER = Path('../city-action-order.csv')
LANG = 'en'
EXAMPLE_ACTION, EXAMPLE_CITY = 'c40_0048', 'BR CXL'   # the methodology's worked example; Caxias do Sul for the city list
TOP_N = 5          # policy documents and past projects shown in the action detail
PAGE_SIZE = 20     # default page size of the full lists

def read(name, path=None):
    with open(path or DATA / f'{name}.csv', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def num(v):
    if v in ('', None): return None
    x = float(v)
    return int(x) if x.is_integer() else round(x, 4)

def text(v): return v if v not in ('', None) else None
def flag(v): return {'true': True, 'false': False}.get(str(v).lower())

def by(rows, key='action_id'):
    out = {}
    for r in rows: out.setdefault(r[key], []).append(r)
    return out

actions = {r['action_id']: r for r in read('adaptation-action')}
impact = by(read('actions-impact'))
legal = {r['action_id']: r for r in read('actions-legal-feasibility')}
norms = by(read('actions-legal-norms'))
cost = {r['action_id']: r for r in read('actions-financial-cost')}
cobenefits = by(read('actions-cobenefits'))
policy = {r['action_id']: r for r in read('actions-policy-alignment')}
evidence = by(read('actions-policy-evidence'))
documents = {r['doc_id']: r for r in read('policy-documents')}
projects = by(read('actions-projects'))
programmes = {r['programme_id']: r for r in read('finance-programs')}
action_programmes = by(read('actions-finance-programs'))
relations = read('actions-relations')
finance_profile = {r['locode']: r for r in read('city-finance-profile')}
order = by(read('city-action-order', ORDER), 'locode')

COBENEFIT_KEY = {
    'Social equity and community capacity': 'social_equity', 'Public health': 'public_health',
    'Affordable and adequate housing': 'housing', 'Air quality': 'air_quality', 'Water quality': 'water_quality',
    'Biodiversity, habitats and ecosystem services': 'biodiversity',
    'Mobility and access to opportunities and services': 'mobility', 'Local economic development': 'local_economy',
}

# Relations: (action, related, type) means `related` is a pre- or co-action of `action`
related_of = {}
for r in relations:
    related_of.setdefault(r['action_id'], []).append(r)

SUPPORTING = {i for i in actions if not any(r['eligibility'] == 'Direct' for r in impact[i])}   # no Direct link: not ordered
assert len(SUPPORTING) == 9

def name(i, lang=LANG): return actions[i][f'action_name_{lang}']
def description(i, lang=LANG): return text(actions[i][f'description_{lang}']) or actions[i]['description_en']
def direct_sectors(i): return sorted({r['sector'] for r in impact[i] if r['eligibility'] == 'Direct'})
def cobenefit_scores(i): return {COBENEFIT_KEY[r['co_benefit']]: num(r['score']) for r in cobenefits[i]}

def meta(endpoint, **extra):
    return {'endpoint': endpoint, 'country_code': 'BR', 'mock': True, 'lang': LANG, **extra}

def write(name, payload):
    Path(f'{name}.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'{name}.json', round(Path(f'{name}.json').stat().st_size / 1024), 'KB')

## City action list

The actions ordered for one city (29 in each test city). Each row carries:

- **position**: place in the default order. Null for an action with a Direct link to no risk scored for the city; these come last (none in the test cities).
- **scores**: every part of the ordering, all 0 to 1.
- **impact_risk**: the city risk cell where the action's impact is highest.
- **attributes**: timeline, cost band, legal grade, policy grade, the sectors the action has a Direct link to, and the co-benefit scores.

In [2]:
def scores(o):
    return {'overall': num(o['overall_score']), 'impact': num(o['impact_score']), 'feasibility': num(o['feasibility_score']),
            'legal': num(o['legal_score_scaled']), 'cost': num(o['cost_score']), 'policy_alignment': num(o['alignment_score'])}

def city_row(o):
    i = o['action_id']
    return {
        'action_id': i,
        'name': name(i),
        'description': description(i),
        'position': num(o['position']),
        'scores': scores(o),
        'impact_risk': {'sector': o['impact_sector'], 'risk': o['impact_risk']} if o['impact_risk'] else None,
        'timeline': actions[i]['timeline'],
        'cost_band': cost[i]['cost_band'],
        'legal_grade': legal[i]['legal_grade'],
        'policy_grade': policy[i]['grade'],
        'risk_sectors': direct_sectors(i),
        'co_benefits': cobenefit_scores(i),
        'counts': {
            'policy_documents': len(evidence.get(i, [])),
            'finance_programmes': len(action_programmes.get(i, [])),
            'past_projects': len(projects.get(i, [])),
            'supporting_actions': len({r['related_action_id'] for r in related_of.get(i, []) if r['related_action_id'] in SUPPORTING}),
        },
    }

CHECKS = ['env_body', 'env_council', 'env_council_budget', 'env_fund', 'env_earmarked_resources', 'climate_legislation',
          'climate_training', 'civil_defence_body', 'civil_defence_budget', 'planning_instruments']

def city_context(locode):
    f = finance_profile[locode]
    return {
        'locode': locode, 'city': f['city'], 'state': f['state'],
        'finance_context': {
            'credit': {'capag': f['capag'], 'credit_screen': f['credit_screen'],
                       'fiscal_base_year': num(f['capag_fiscal_base_year']), 'release_date': f['capag_release_date'],
                       'source': 'STN CAPAG Municipios (1 Jun 2026)'},
            'preparation_capacity': {
                'score': num(f['preparation_capacity_score']), 'max_score': num(f['preparation_capacity_max']),
                'national_median': num(f['national_median']), 'below_national_median': flag(f['below_national_median']),
                'checks': {k: num(f[k]) for k in CHECKS}, 'source': 'IBGE MUNIC 2020 and 2021'},
        },
    }

for locode in [EXAMPLE_CITY]:
    rows = order[locode]
    write(f"city-adaptation-actions-{locode.replace(' ', '-')}", {
        'meta': meta('GET /api/v1/cities/{locode}/adaptation-actions', locode=locode, total_records=len(rows)),
        'data': {'city': city_context(locode), 'actions': [city_row(o) for o in rows]},
    })

city-adaptation-actions-BR-CXL.json 51 KB


## Action detail

Everything behind one action. It is the same for every city. Co-benefits sit on their own, not under alignment, because they are not scored.

`supporting_actions` lists the supporting actions linked to this action, each marked `pre_action` (comes first) or `co_action` (goes with it). `related_actions` lists its pre- and co-actions that are themselves ordered actions.

Policy documents and past projects show the top 5 and the total. The full lists come from their own paged endpoints (next section).

- **Policy documents:** one row per document with that document's best passage, strongest grade first, then by how much the document adds to support.
- **Past projects:** high matches first, then the most recent decision.

In [3]:
def impact_link(r):
    return {
        'sector': r['sector'], 'component': r['component'].lower(), 'eligibility': r['eligibility'].lower(),
        'effectiveness': text(r['effectiveness'].lower()), 'confidence': text(r['confidence'].lower()),
        'maladaptation_signal': {'Maladaptation signal': True, 'No maladaptation signal': False}.get(r['maladaptation']),
        'component_score': num(r['component_score']), 'human_reviewed': flag(r['human_reviewed']),
        'eligibility_rationale': text(r['eligibility_rationale']), 'effectiveness_rationale': text(r['effectiveness_rationale']),
        'confidence_sources': text(r['confidence_sources']), 'maladaptation_rationale': text(r['maladaptation_rationale']),
    }

def sorted_documents(i):
    return sorted(evidence.get(i, []), key=lambda r: (['Strong', 'Medium', 'Weak'].index(r['grade']), -float(r['adds_to_support'] or 0), r['doc_id']))

def sorted_projects(i):
    rows = sorted(projects.get(i, []), key=lambda r: r['decision_date'] or '', reverse=True)            # newest first
    return sorted(rows, key=lambda r: {'High': 0, 'Medium': 1}.get(r['match_strength'], 2))           # then high matches first (stable)

def document_item(d):
    return {
        'doc_id': d['doc_id'], 'title': d['document'], 'tier': d['tier'], 'link': text(documents[d['doc_id']]['link']),
        'grade': d['grade'], 'action_match': text(d['action_match']), 'policy_strength': text(d['policy_strength']),
        'pdf_page': num(d['pdf_page']), 'text_pt': text(d['decisive_text_pt']), 'reason': text(d['reason']),
        'adds_to_support': num(d['adds_to_support']),
    }

def project_item(r):
    return {
        'project_reference': r['project_reference'], 'name': r['project_name'], 'name_pt': text(r['project_name_pt']),
        'record_type': text(r['record_type']), 'recipient': text(r['recipient']), 'location': text(r['location']),
        'instrument': text(r['finance_instrument']), 'approved_amount': num(r['approved_amount']),
        'disbursed_amount': num(r['disbursed_amount']), 'currency': text(r['currency']), 'status': text(r['status']),
        'decision_date': text(r['decision_date']), 'dataset': text(r['dataset']), 'match_strength': text(r['match_strength']),
        'why_it_matches': text(r['why_it_matches']),
    }

def detail(i):
    c, p = cost[i], policy[i]
    docs, projs = sorted_documents(i), sorted_projects(i)
    return {
        'action_id': i, 'source': actions[i]['source'], 'name': name(i), 'description': description(i),
        'timeline': actions[i]['timeline'], 'hazards': actions[i]['hazards'].split('|') if actions[i]['hazards'] else [],
        'impact': {
            'has_direct_link': any(r['eligibility'] == 'Direct' for r in impact[i]),
            'links': [impact_link(r) for r in impact[i] if r['eligibility'] != 'Out of Scope'],
        },
        'feasibility': {
            'legal': {
                'grade': legal[i]['legal_grade'], 'score': num(legal[i]['legal_score']),
                'norms': [{
                    'norm_id': n['norm_id'], 'name': n['norm_name'], 'code': n['norm_code'], 'type': n['norm_type'],
                    'description_pt': text(n['norm_description']), 'authority': n['authority'], 'competence': n['competence'],
                    'validation': n['validation'], 'validation_date': n['validation_date'], 'link': text(n['norm_link']),
                } for n in norms.get(i, [])],
            },
            'cost': {
                'band': c['cost_band'], 'index': num(c['cost_index']),
                'components': {k: num(c[k]) for k in ('technical_complexity', 'resource_intensity', 'spatial_scale', 'maintenance', 'implementation_time')},
            },
        },
        'alignment': {
            'policy': {
                'grade': p['grade'], 'score': num(p['score']), 'document_support': num(p['document_support']),
                'explanation': text(p['explanation']),
                'documents': {'total': len(docs), 'showing': min(len(docs), TOP_N), 'items': [document_item(d) for d in docs[:TOP_N]]},
            },
        },
        'co_benefits': [{
            'key': COBENEFIT_KEY[r['co_benefit']], 'name': r['co_benefit'], 'score': num(r['score']),
            'justification': text(r['justification']), 'review_status': text(r['review_status']),
            'reviewer_notes': text(r['reviewer_notes']),
        } for r in cobenefits[i]],
        'supporting_actions': [{'action_id': r['related_action_id'], 'name': name(r['related_action_id']), 'relation_type': r['relation_type']}
                               for r in related_of.get(i, []) if r['related_action_id'] in SUPPORTING],
        'related_actions': [{'action_id': r['related_action_id'], 'name': name(r['related_action_id']), 'relation_type': r['relation_type']}
                            for r in related_of.get(i, []) if r['related_action_id'] not in SUPPORTING],
        'finance': {
            'programmes': [{
                'programme_id': g['programme_id'],
                **{k: text(programmes[g['programme_id']][k]) for k in ('programme', 'kind', 'source', 'channel', 'instrument', 'recipient', 'status')},
                'budget': None if not programmes[g['programme_id']]['budget_amount'] else {
                    'amount': num(programmes[g['programme_id']]['budget_amount']),
                    'unit': programmes[g['programme_id']]['budget_unit'], 'period': programmes[g['programme_id']]['budget_period'],
                },
                'named_in': {'doc_id': g['doc_id'], 'title': documents[g['doc_id']]['title'], 'pdf_page': num(g['pdf_page']),
                             'policy_grade': g['best_policy_grade']},
            } for g in action_programmes.get(i, [])],
            'past_projects': {'total': len(projs), 'showing': min(len(projs), TOP_N), 'items': [project_item(r) for r in projs[:TOP_N]]},
        },
    }

write(f'adaptation-action-{EXAMPLE_ACTION}', {
    'meta': meta('GET /api/v1/adaptation-actions/{action_id}', top_n=TOP_N),
    'data': detail(EXAMPLE_ACTION),
})

adaptation-action-c40_0048.json 27 KB


## Full lists

The action detail shows the top 5 policy documents and past projects. These endpoints return all of them, in the same order, a page at a time. `page` starts at 1.

In [4]:
def page_of(rows, page=1, page_size=PAGE_SIZE):
    total = len(rows)
    return {'page': page, 'page_size': page_size, 'total_records': total, 'total_pages': max(1, -(-total // page_size))}, \
           rows[(page - 1) * page_size: page * page_size]

paging, rows = page_of(sorted_documents(EXAMPLE_ACTION))
write(f'adaptation-action-{EXAMPLE_ACTION}-policy-documents', {
    'meta': meta('GET /api/v1/adaptation-actions/{action_id}/policy-documents', action_id=EXAMPLE_ACTION, **paging),
    'data': [document_item(d) for d in rows],
})

paging, rows = page_of(sorted_projects(EXAMPLE_ACTION))
write(f'adaptation-action-{EXAMPLE_ACTION}-past-projects', {
    'meta': meta('GET /api/v1/adaptation-actions/{action_id}/past-projects', action_id=EXAMPLE_ACTION, **paging),
    'data': [project_item(r) for r in rows],
})

adaptation-action-c40_0048-policy-documents.json 11 KB
adaptation-action-c40_0048-past-projects.json 17 KB


In [5]:
# Check: every supporting action shows up in the detail of at least one ordered action
shown = {r['related_action_id'] for r in relations if r['related_action_id'] in SUPPORTING and r['action_id'] not in SUPPORTING}
print('supporting actions shown under an ordered action:', len(shown), 'of', len(SUPPORTING), sorted(SUPPORTING - shown) or '')

# Checks: every field in the responses is defined in api-fields.csv, and every defined field appears
fields = list(csv.DictReader(open('api-fields.csv', encoding='utf-8')))
defined = {(f['endpoint'], f['field']) for f in fields}

def paths(obj, prefix=''):
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{prefix}.{k}' if prefix else k
            yield p
            yield from paths(v, p)
    elif isinstance(obj, list) and obj and isinstance(obj[0], dict):
        for v in obj: yield from paths(v, prefix + '[]')

found = set()
for f, ep in [('city-adaptation-actions-BR-CXL.json', 'city list'), ('adaptation-action-c40_0048.json', 'action detail'),
              ('adaptation-action-c40_0048-policy-documents.json', 'policy documents'),
              ('adaptation-action-c40_0048-past-projects.json', 'past projects')]:
    data = json.load(open(f))['data']
    found |= {(ep, p) for p in paths(data)}
print('in responses, not defined:', sorted(found - defined) or 'none')
print('defined, not in responses:', sorted(defined - found) or 'none')

supporting actions shown under an ordered action: 8 of 9 ['icare_0164']
in responses, not defined: none
defined, not in responses: none
